# Browser Tool로 유료 Content에 액세스하는 Strands Agent

> **Pattern 참고 자료:** 이 Notebook은 browser + payment 아키텍처와 코드 pattern을 보여 줍니다. HTTP 402 Payment Required를 반환하는 x402 지원 content endpoint가 없으면 end-to-end로 실행할 수 없습니다. 향후 use case sample에서는 이 flow를 end-to-end로 테스트할 수 있도록 로컬 또는 AWS에서 실행 가능한 dummy x402 paywall server를 제공할 예정입니다.

## 개요

Tutorial 01에서는 `AgentCorePaymentsPlugin`이 tool output level에서 x402 payment를 처리했습니다. 이는 plugin에서 intercept하고 retry할 수 있는 API endpoint에 적합합니다. 하지만 **browser에서 rendering되는 content**(paywall article, content site)의 경우 agent가 동일한 Playwright session 내에서 402를 감지하고 결제한 후 proof header와 함께 retry해야 합니다.

이 튜토리얼에서는 다음 항목을 사용하는 custom `browse_with_payment` tool을 구축합니다.
- Managed cloud Chromium을 위한 **AgentCore Browser**(`BrowserClient`)
- Browser automation library인 **Playwright**(WebSocket을 통해 AgentCore Browser에 연결)
- x402 signing을 위한 **AgentCore payments**(`PaymentManager.generate_payment_header()`)

### Plugin 대신 Custom Tool을 사용하는 이유

Tutorial 01의 `AgentCorePaymentsPlugin`은 tool output level에서 response를 intercept하고 payment에 sign한 다음 외부에서 tool 호출을 retry하여 402를 처리합니다. 이는 API endpoint에 적합합니다. 하지만 browser에서 rendering되는 content의 경우 402가 browser session 내부에서 발생하며 cookie, auth token, DOM context를 유지하려면 동일한 session에서 retry해야 합니다. Plugin은 이를 수행할 수 없으므로 tool이 내부에서 payment flow를 처리해야 합니다.

### 아키텍처

```
Strands Agent
  └── browse_with_payment tool
        │
        ├── 1. BrowserClient.start() → managed cloud Chromium
        ├── 2. Playwright connects to AgentCore Browser (WebSocket)
        ├── 3. page.goto(url) → response interceptor detects 402
        ├── 4. Extract x402 requirements from response
        ├── 5. PaymentManager.generate_payment_header() → signed proof
        ├── 6. page.route() injects proof header
        ├── 7. page.goto(url) retries → 200 + content
        └── 8. Return content to agent
```

Retry를 위해 browser session을 계속 열어 두어야 하므로 payment logic은 tool 내부에 있습니다. Plugin이 외부에서 retry를 처리하는 Tutorial 01과 다른 점입니다.

> **Testnet 전용입니다.** 모든 코드는 [faucet.circle.com](https://faucet.circle.com/)의 무료 USDC와 함께 Base Sepolia 또는 Solana Devnet을 사용합니다. Testnet USDC에는 실제 가치가 없습니다.


### 아키텍처

![Architecture](images/architecture.png)


### 튜토리얼 세부 정보

| 항목                | 세부 정보                                                                |
|:--------------------|:------------------------------------------------------------------------|
| Tutorial type       | Task 기반                                                               |
| Agent type          | Single                                                                  |
| Agentic Framework   | Strands Agents                                                          |
| LLM model           | Anthropic Claude Sonnet                                                 |
| Tutorial 구성 요소  | AgentCore Browser, Playwright, AgentCore payments, x402                 |
| 예제 난이도         | 중급                                                                    |
| 사용 SDK            | bedrock-agentcore SDK (BrowserClient + PaymentManager), Strands, Playwright |

## 사전 요구 사항

* Tutorial 00 및 01 완료(`.env` 존재)
* https://faucet.circle.com/의 testnet USDC를 Wallet에 입금
* `pip install -r requirements.txt`
* `python -m playwright install chromium`

이 튜토리얼은 Tutorial 00에서 구성한 Coinbase CDP 또는 Stripe(Privy) wallet provider 모두에서 작동합니다. AWS credentials에는 Tutorial 00에서 생성한 IAM 권한(`setup_payment_roles()`)이 필요합니다.

In [ ]:
%pip install -r requirements.txt --quiet
!python -m playwright install chromium

## 1단계 — Config 불러오기

In [ ]:
import sys
import os
import json

sys.path.append("..")

from dotenv import load_dotenv

load_dotenv(override=True)

from utils import load_tutorial_env, print_summary

config = load_tutorial_env()

PAYMENT_MANAGER_ARN = config["payment_manager_arn"]
REGION = config["region"]
USER_ID = config["user_id"]

if config.get("multi_provider"):
    PROVIDER = list(config["instruments"].keys())[0]
    INSTRUMENT_ID = config["instruments"][PROVIDER]["instrument_id"]
    CONNECTOR_ID = config["instruments"][PROVIDER]["connector_id"]
else:
    INSTRUMENT_ID = config["instrument_id"]
    CONNECTOR_ID = config["connector_id"]
    PROVIDER = config.get("provider_type", "unknown")

MODEL_ID = os.environ.get("MODEL_ID", "us.anthropic.claude-sonnet-4-6")

print_summary(
    "Config",
    payment_manager_arn=PAYMENT_MANAGER_ARN,
    provider=PROVIDER,
    instrument_id=INSTRUMENT_ID,
)

## 2단계 — Payment Session 생성

In [ ]:
from bedrock_agentcore.payments import PaymentManager

# Tutorial 00의 기존 Payment Manager ARN을 래핑하는 SDK client
manager = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)

# Instrument가 ACTIVE인지 검증
instr = manager.get_payment_instrument(user_id=USER_ID, payment_instrument_id=INSTRUMENT_ID)
instr_status = instr.get("status", "UNKNOWN")
assert instr_status == "ACTIVE", f"Instrument is {instr_status} — fund and delegate in Tutorial 00/03 first"

# 이 task를 위한 새 session 생성
session_resp = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "1.00", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
SESSION_ID = session_resp["paymentSessionId"]
print(f"✅ Instrument {INSTRUMENT_ID} is {instr_status}")
print(f"✅ Session: {SESSION_ID} (budget: $1.00, expiry: 60 min)")

## 3단계 — `browse_with_payment` Tool 구축

이 부분이 튜토리얼의 핵심입니다. Payment flow가 browser session 내부에서 이루어져야 하므로 아래 `@tool`은 custom tool입니다(plugin은 retry 간에 browser state를 유지할 수 없음). 이 tool은 다음 작업을 수행합니다.
1. AgentCore Browser(`BrowserClient`)를 통해 managed browser session 시작
2. WebSocket을 통해 Playwright를 managed browser에 연결
3. URL로 이동
4. Response가 402이면 x402 requirement 추출
5. `generate_payment_header()`를 호출하여 payment에 sign
6. Proof header를 삽입하고 동일한 session에서 retry
7. Content를 agent에 반환

In [ ]:
# PaymentManager는 2단계에서 이미 생성했으며 아래 tool 내부에서 사용
print("✅ PaymentManager ready (from Step 2)")

## 4단계 — `browse_with_payment` Tool 구축

이 부분이 튜토리얼의 핵심입니다. Payment flow가 browser session 내부에서 이루어져야 하므로 아래 `@tool`은 custom tool입니다(plugin은 retry 간에 browser state를 유지할 수 없음). 이 tool은 다음 작업을 수행합니다.
1. AgentCore Browser(`BrowserClient`)를 통해 managed browser session 시작
2. WebSocket을 통해 Playwright를 managed browser에 연결
3. URL로 이동
4. Response가 402이면 x402 requirement 추출
5. `generate_payment_header()`를 호출하여 payment에 sign
6. Proof header를 삽입하고 동일한 session에서 retry
7. Content를 agent에 반환

### Payment Flow 순서

![Payment Flow](images/payment_flow.png)


In [ ]:
import asyncio
from playwright.async_api import async_playwright
from bedrock_agentcore.tools.browser_client import BrowserClient
from strands import tool


def extract_x402_requirements(headers, body):
    """402 응답에서 x402 결제 요구 사항을 파싱합니다."""
    # Body를 JSON으로 parsing 시도(x402 v2 format)
    try:
        return json.loads(body)
    except (json.JSONDecodeError, TypeError):
        pass
    # 실패하면 header 사용
    return {"headers": headers, "body": body}


def _format_result(status: int, content: str, paid: bool, url: str) -> str:
    """Strands에서 사용할 수 있도록 도구 결과를 하나의 텍스트 문자열로 만듭니다."""
    header = f"URL: {url}\nHTTP status: {status}\nPaid: {paid}"
    return f"{header}\n\n{content[:5000]}"


@tool
def browse_with_payment(url: str) -> str:
    """Navigate to a URL using a managed cloud browser. If the endpoint returns
    402 Payment Required, automatically pay via AgentCore payments and retry.

    Uses AgentCore Browser (managed Chromium) + Playwright for navigation.
    Payment is signed via PaymentManager.generate_payment_header().

    Args:
        url: The URL to navigate to and retrieve content from.

    Returns:
        A text summary containing the URL, HTTP status, paid flag, and page content.
    """

    async def _browse():
        browser_client = BrowserClient(region=REGION)
        try:
            browser_client.start()
            ws_url, ws_headers = browser_client.generate_ws_headers()
            print("  🌐 Browser session started")

            async with async_playwright() as pw:
                browser = await pw.chromium.connect_over_cdp(
                    endpoint_url=ws_url,
                    headers=ws_headers,
                    timeout=30000,
                )
                context = browser.contexts[0] if browser.contexts else await browser.new_context()
                page = context.pages[0] if context.pages else await context.new_page()

                # 첫 번째 navigation
                response = await page.goto(url, wait_until="domcontentloaded", timeout=30000)
                status = response.status if response else 0
                print(f"  → HTTP {status}")

                # 402인 경우 requirement 추출, 결제, retry
                if status == 402:
                    print("  💰 402 Payment Required — processing payment...")
                    body = await response.text()
                    resp_headers = await response.all_headers()

                    # AgentCore를 통해 payment proof 생성
                    payment_header = manager.generate_payment_header(
                        user_id=USER_ID,
                        payment_instrument_id=INSTRUMENT_ID,
                        payment_session_id=SESSION_ID,
                        payment_required_request={
                            "statusCode": 402,
                            "headers": resp_headers,
                            "body": body,
                        },
                    )
                    print("  ✅ Payment signed")

                    # Main navigation request에만 payment header를 삽입하여
                    # sub-resource(image, CSS, beacon)에 X-PAYMENT가 노출되지 않도록 함
                    async def add_payment_headers(route, request):
                        if request.is_navigation_request():
                            headers = {**request.headers, **payment_header}
                            await route.continue_(headers=headers)
                        else:
                            await route.continue_()

                    await page.route("**/*", add_payment_headers)
                    response = await page.goto(url, wait_until="domcontentloaded", timeout=30000)
                    status = response.status if response else 0
                    print(f"  → Retry: HTTP {status}")

                    if status == 200:
                        content = await page.inner_text("body")
                        await browser.close()
                        return _format_result(200, content, paid=True, url=url)
                    else:
                        await browser.close()
                        return _format_result(
                            status,
                            f"Payment retry failed with HTTP {status}",
                            paid=True,
                            url=url,
                        )

                # Payment 불필요
                content = await page.inner_text("body")
                await browser.close()
                return _format_result(status, content, paid=False, url=url)

        finally:
            browser_client.stop()
            print("  🌐 Browser session closed")

    return asyncio.run(_browse())


print("✅ browse_with_payment tool created")
print("   Uses: BrowserClient + Playwright + PaymentManager.generate_payment_header()")

## 5단계 — Agent 생성

Strands agent에 `browse_with_payment` tool을 제공합니다. Agent가 탐색 시점을 결정하며 tool은 내부에서 payment flow를 처리합니다.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

SYSTEM_PROMPT = """You are a content retrieval agent with browser access and payment capabilities.

Use the browse_with_payment tool to navigate to URLs and retrieve content.
If a page requires payment, the tool handles it automatically.
Summarize the content you retrieve.
Always report what you paid and what content you received."""

agent = Agent(
    model=BedrockModel(model_id=MODEL_ID, streaming=True),
    tools=[browse_with_payment],
    system_prompt=SYSTEM_PROMPT,
)

print("✅ Agent created with browse_with_payment tool")

## 6단계 — Agent가 유료 Endpoint 탐색

Agent는 managed browser를 통해 x402 지원 endpoint로 이동합니다. Tool은 402를 감지하고 payment에 sign한 후 proof header와 함께 retry하여 content를 반환합니다.

In [ ]:
import time

TARGET_URL = "https://api.cdp.coinbase.com/platform/v2/x402/discovery/search?query=technology+trends&limit=3"

print(f"🌐 Target: {TARGET_URL}")
print("💰 Budget: $1.00 USD")
print("🖥️  Browser: AgentCore managed Chromium\n")

start = time.time()
result = agent(f"Browse to this URL and retrieve the content: {TARGET_URL}\nSummarize what you find.")
elapsed = time.time() - start

print(f"\n{'=' * 60}")
print(f"  Completed in {elapsed:.1f}s")
print(f"{'=' * 60}")
print(result.message)

## 7단계 — Session 지출 검증

In [ ]:
session_info = manager.get_payment_session(
    user_id=USER_ID,
    payment_session_id=SESSION_ID,
)
sess = session_info
print_summary(
    "Session Spend",
    session_id=SESSION_ID,
    available=sess.get("availableLimits", {}).get("availableSpendAmount", "N/A"),
    budget_limit=sess.get("limits", {}).get("maxSpendAmount", "N/A"),
)

## Payment Trace 보기

모든 payment에서 trace가 생성됩니다. CloudWatch GenAI Observability Dashboard에서 확인하세요.


In [ ]:
print("🔍 View your agent traces: CloudWatch → GenAI Observability Dashboard")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core")

## 요약

x402 payment를 처리하는 custom browser tool이 포함된 Strands agent를 구축했습니다.

1. **BrowserClient** — managed cloud Chromium session 시작(AgentCore Browser)
2. **Playwright** — WebSocket을 통해 AgentCore Browser에 연결하고 x402 endpoint로 이동
3. **402 감지** — response interceptor가 payment requirement 감지
4. **Payment signing** — `PaymentManager.generate_payment_header()`가 proof 생성
5. **Proof와 함께 retry** — `page.route()`가 header를 삽입하고 동일한 browser session에서 retry
6. **Content 반환** — agent가 유료 content를 받아 요약

### 두 Payment Pattern 비교

| Pattern | Tool | Payment 처리 | 적합한 용도 |
|---------|------|-----------------|----------|
| **Plugin(Tutorial 01)** | `http_request` | Plugin이 tool output을 intercept하고 외부에서 retry | API endpoint, MCP tool |
| **Browser(이 튜토리얼)** | Custom `browse_with_payment` | Tool이 내부에서 402를 처리하고 동일한 session에서 retry | Browser에서 rendering되는 content, paywall |

API 호출에는 plugin pattern을 사용합니다. Payment retry 간에 session state(cookie, auth token, DOM context)를 유지해야 할 때는 browser pattern을 사용합니다.

### 배포된 Agent의 Role 분리

이 Notebook은 AWS credentials로 로컬에서 실행됩니다. 배포 시 runtime process는 ProcessPaymentRole로 실행되며, plugin은 app backend에서 설정한 budget 내에서 agent를 대신해 `ProcessPayment`를 호출합니다. Runtime은 session 생성, limit 수정 또는 wallet provision을 할 수 없습니다. Agent(LLM)는 `ProcessPayment`를 직접 호출하지 않습니다. App backend가 invocation payload를 통해 모든 payment context를 전달합니다. 전체 구현은 Tutorial 02를 참조하세요.

Role 분리를 로컬에서 테스트하려면 assumed-role session을 SDK client에 전달합니다.

```python
from utils import assume_role
import boto3

# 앱 백엔드(ManagementRole)가 세션 생성
manager = PaymentManager(payment_manager_arn=ARN, region_name=REGION)
session = manager.create_payment_session(user_id=USER_ID, ...)

# Agent는 ProcessPaymentRole로 실행되며 ProcessPayment만 수행 가능
agent_session = assume_role(boto3.Session(), PROCESS_PAYMENT_ROLE_ARN, 'agent')
agent_manager = PaymentManager(
    payment_manager_arn=ARN, boto3_session=agent_session
)
# 제한된 자격 증명으로 generate_payment_header()에 agent_manager 전달
```

## 리소스 정리

**비용 안내:** AgentCore Browser session과 payment session에는 사용량에 따라 AWS 요금이 발생할 수 있습니다. 이 튜토리얼에서는 실제 가치가 없는 testnet USDC를 사용하지만 AWS infrastructure에는 요금이 부과됩니다.

Browser session은 구성된 timeout 후 자동으로 만료됩니다. Payment session은 구성된 `expiryTimeInMinutes` 후 만료됩니다. 이 튜토리얼에서는 수동으로 리소스를 정리할 필요가 없습니다.

# 축하합니다!

AgentCore Browser + AgentCore payments를 사용하여 web content에 결제하는 browser agent를 구축했습니다. 다음 과정: **Tutorial 06** — Payment Memory를 사용하는 Research Agent